In [1]:
import torch
from pathlib import Path
from scipy.constants import pi
from ase.io import read
import numpy as np

from graph_longrange.kspace import compute_k_vectors_flat
from graph_longrange.energy import GTOElectrostaticEnergy
from graph_longrange.gto_utils import gto_basis_kspace_cutoff
from ase.build import bulk


torch.set_default_dtype(torch.float64)


/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  _Jd, _W3j_flat, _W3j_indices = torch.load

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/tw/miniforge3/envs/surfpes/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/Users/tw/miniforge3/envs/surfpes/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/Users/tw/miniforge3/envs/surfpes/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wro

In [2]:
bulk_nacl = bulk("NaCl", crystalstructure="rocksalt", a=5.64)
print(bulk_nacl.get_chemical_symbols())
bulk_nacl.arrays['oxi_states'] = np.array([+1., -1.])

['Na', 'Cl']


In [3]:
def atoms_list_to_batch(atoms_list):
    positions = []
    batch = []
    cells = []
    volumes = []
    pbcs = []

    for graph_i, atoms in enumerate(atoms_list):
        pos = torch.tensor(atoms.get_positions(), dtype=torch.get_default_dtype())
        positions.append(pos)
        batch.append(torch.full((pos.shape[0],), graph_i, dtype=torch.long))

        cell = torch.tensor(atoms.cell.array, dtype=torch.get_default_dtype())
        if torch.allclose(cell, torch.zeros_like(cell)):
            cell = torch.eye(3, dtype=cell.dtype)
        cells.append(cell)
        volumes.append(abs(torch.det(cell)))
        pbcs.append(torch.tensor(atoms.pbc, dtype=torch.bool))

    return (
        torch.cat(positions, dim=0),
        torch.cat(batch, dim=0),
        torch.stack(cells, dim=0),
        torch.stack(volumes, dim=0),
        torch.stack(pbcs, dim=0),
    )

In [4]:
# density_max_l is the multipole order. 0=charges, 1=charges+dipoles, etc...
# the realspace evalation is currently supported for only l<=1. 
density_max_l = 0

# this is sigma_n in the GTO basis, we generally use quite wide gaussians in ML models
density_smearing_width = 0.25

# use this function for a heuristic estimate of the k-space cutoff, 
# which is needed to determine the number of k-vectors to use in the reciprocal space sum.
kspace_cutoff = 3 * gto_basis_kspace_cutoff(
    sigmas=[density_smearing_width],
    max_l=density_max_l,
)

energy_block = GTOElectrostaticEnergy(
    density_max_l=density_max_l,
    density_smearing_width=density_smearing_width,
    kspace_cutoff=kspace_cutoff,
    include_self_interaction=False,
)

In [5]:
positions_ttf, batch_ttf, cell_ttf, volume_ttf, pbc_ttf = atoms_list_to_batch([bulk_nacl])

In [6]:
r_cell = 2 * pi * torch.linalg.inv(cell_ttf).transpose(-1, -2)
k_vectors, k_norm2, k_vector_batch, k0_mask = compute_k_vectors_flat(
    cutoff=kspace_cutoff,
    cell_vectors=cell_ttf,
    r_cell_vectors=r_cell,
)

In [7]:
multipoles = bulk_nacl.arrays['oxi_states']
multipoles = torch.tensor(multipoles)
multipoles = multipoles.view(-1, 1)

In [8]:
energy_bulk = energy_block(
    k_vectors=k_vectors,
    k_norm2=k_norm2,
    k_vector_batch=k_vector_batch,
    k0_mask=k0_mask,
    source_feats=multipoles,
    node_positions=positions_ttf,
    batch=batch_ttf,
    volume=volume_ttf,
    pbc=pbc_ttf,
)

energy_bulk

tensor([-8.9235])

In [9]:
from pymatgen.analysis.energy_models import EwaldElectrostaticModel
from pymatgen.core import Structure

model = EwaldElectrostaticModel()
model.get_energy(Structure.from_ase_atoms(bulk_nacl))

np.float64(-8.923514389332672)

In [10]:
from ase.build import bulk, surface

slab_nacl = surface(bulk_nacl, (1, 1, 1), layers=20, vacuum=10)
slab_nacl.set_pbc([True, True, True])

multipoles_slab = torch.tensor(slab_nacl.arrays['oxi_states']).view(-1, 1)

In [11]:
positions_slab, batch_slab, cell_slab, volume_slab, pbc_slab = atoms_list_to_batch([slab_nacl])

r_cell = 2 * pi * torch.linalg.inv(cell_slab).transpose(-1, -2)
k_vectors, k_norm2, k_vector_batch, k0_mask = compute_k_vectors_flat(
    cutoff=kspace_cutoff,
    cell_vectors=cell_slab,
    r_cell_vectors=r_cell,
)

energy_slab = energy_block(
    k_vectors=k_vectors,
    k_norm2=k_norm2,
    k_vector_batch=k_vector_batch,
    k0_mask=k0_mask,
    source_feats=multipoles_slab,
    node_positions=positions_slab,
    batch=batch_slab,
    volume=volume_slab,
    pbc=pbc_slab,
)

energy_slab

tensor([-155.3756])

In [12]:
model.get_energy(Structure.from_ase_atoms(slab_nacl))

np.float64(-155.37555399168457)

In [13]:
def get_ewald_energy(structure: Structure, pbc: list[bool]):
    atoms = structure.to_ase_atoms()
    atoms.set_pbc(pbc)

    multipoles = atoms.arrays['oxi_states']
    multipoles = torch.tensor(multipoles).view(-1, 1)


    positions, batch, cell, volume, pbc = atoms_list_to_batch([atoms])

    r_cell = 2 * pi * torch.linalg.inv(cell).transpose(-1, -2)
    k_vectors, k_norm2, k_vector_batch, k0_mask = compute_k_vectors_flat(
        cutoff=kspace_cutoff,
        cell_vectors=cell,
        r_cell_vectors=r_cell,
)

    energy = energy_block(
        k_vectors=k_vectors,
        k_norm2=k_norm2,
        k_vector_batch=k_vector_batch,
        k0_mask=k0_mask,
        source_feats=multipoles,
        node_positions=positions,
        batch=batch,
        volume=volume,
        pbc=pbc,
    )

    return energy

nacl_bulk_struct = Structure.from_ase_atoms(bulk_nacl)
print(get_ewald_energy(nacl_bulk_struct, [True, True, True]))
print(get_ewald_energy(Structure.from_ase_atoms(slab_nacl), [True, True, True]))

tensor([-8.9235])
tensor([-155.3756])


In [14]:
from pymatgen.core import Structure
from joblib import Memory
from mp_api.client import MPRester
from ase.visualize import view
import numpy as np
from pymatgen.core.surface import SlabGenerator

memory = Memory('.cachedir')

def view_structure(structure, viewer='ase'):
    return view(structure.to_ase_atoms(), viewer=viewer)

@memory.cache
def get_structure(mp_id):
    with MPRester() as mpr:
        # docs = mpr.materials.summary.search(material_ids=[mp_id], fields=["structure"])
        # structure = docs[0].structure
        # # -- Shortcut for a single Materials Project ID:
        structure = mpr.get_structure_by_material_id(mp_id,)
    
    return structure

structure = get_structure('mp-1265')
structure = structure.to_conventional()


slab_gen = SlabGenerator(structure, [1,1,1], 50, 100, primitive=True, max_normal_search=5) ### If you want the slab to be center set center_slab=True
slab = slab_gen.get_slab()
slab.add_oxidation_state_by_guess()

________________________________________________________________________________
[Memory] Calling __main__--var-folders-v4-9mn3h1611vn857k_73b_dx4w0000gn-T-ipykernel-308475870.get_structure...
get_structure('mp-1265')


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

____________________________________________________get_structure - 1.4s, 0.0min


Structure Summary
Lattice
    abc : 2.9656078098378367 2.9656078098378367 152.54874413762482
 angles : 90.0 90.0 120.00000000000001
 volume : 1161.8945157344754
      A : np.float64(2.9656078098378367) np.float64(0.0) np.float64(1.8159110559221497e-16)
      B : np.float64(-1.4828039049189192) np.float64(2.5682917009810966) np.float64(1.8159110559221497e-16)
      C : np.float64(0.0) np.float64(0.0) np.float64(152.54874413762482)
    pbc : True True True
PeriodicSite: Mg2+ (1.483, 2.568, 1.2e-15) [1.0, 1.0, 5.486e-18]
PeriodicSite: Mg2+ (1.483, 0.8561, 2.421) [0.6667, 0.3333, 0.01587]
PeriodicSite: Mg2+ (-1.443e-15, 1.712, 4.843) [0.3333, 0.6667, 0.03175]
PeriodicSite: Mg2+ (2.966, 0.0, 7.264) [1.0, 0.0, 0.04762]
PeriodicSite: Mg2+ (1.483, 0.8561, 9.686) [0.6667, 0.3333, 0.06349]
PeriodicSite: Mg2+ (-9.992e-16, 1.712, 12.11) [0.3333, 0.6667, 0.07937]
PeriodicSite: Mg2+ (2.966, 0.0, 14.53) [1.0, 0.0, 0.09524]
PeriodicSite: Mg2+ (1.483, 0.8561, 16.95) [0.6667, 0.3333, 0.1111]
PeriodicSit

In [15]:
get_ewald_energy(slab, [True, True, True])

tensor([-607.6706])

In [16]:
model.get_energy(slab)

np.float64(-607.6705821469574)